# Llibreries

In [1]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [2]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")

# Dades d' habitatge
En aquesta topologia ingestarem dades sobre preus dels lloguers (€/m2) i si és possible nombre d' habitatges d' ús turístic.

In [7]:
with ZipFile("../data/raw/habitatge/Taula estadística - Preu mitjà per superfície (€_m²) del lloguer d'habitatges.zip", "r") as zip_ref:
    zip_ref.extractall("../data/raw/habitatge/preu_lloguer_hab_zip_extract")

In [32]:
df_lloguer_raw = pd.read_csv("../data/raw/habitatge/preu_lloguer_hab_zip_extract/Taula estadística.csv")
df_lloguer_raw.head()

,Territori,Tipus de territori,2015,2025
0,el Raval,Barri,"11,1","16,6"
1,el Barri Gòtic,Barri,"11,2","16,6"
2,la Barceloneta,Barri,"16,3","22,6"
3,"Sant Pere, Santa Caterina i la Ribera",Barri,"12,7","18,3"
4,el Fort Pienc,Barri,"10,9","15,8"


In [33]:
df_lloguer_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Territori           73 non-null     object
 1   Tipus de territori  73 non-null     object
 2   2015                73 non-null     object
 3   2025                73 non-null     object
dtypes: object(4)
memory usage: 2.4+ KB


**Observacions:**
- No s' observen nuls a primera instància.
- Formats incorrectes, degut a les comes com a decimals, ho detecta com a texts.

In [34]:
# Eliminem les columnes que no ens interessen
df_lloguer_raw = df_lloguer_raw.drop(columns=["Tipus de territori"])

# Converitm temporalment en format llarg
df_lloguer_stacked = df_lloguer_raw.melt(id_vars=["Territori"], var_name="any", value_name="preu_mitja_m2")
df_lloguer_stacked.head()

,Territori,any,preu_mitja_m2
0,el Raval,2015,"11,1"
1,el Barri Gòtic,2015,"11,2"
2,la Barceloneta,2015,"16,3"
3,"Sant Pere, Santa Caterina i la Ribera",2015,"12,7"
4,el Fort Pienc,2015,"10,9"


In [37]:
# Convertim preu a numèric
df_lloguer_stacked["preu_mitja_m2"] = pd.to_numeric(df_lloguer_stacked["preu_mitja_m2"].astype(str).str.replace(",", "."), errors="coerce")
df_lloguer_stacked.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Territori      146 non-null    object 
 1   any            146 non-null    object 
 2   preu_mitja_m2  145 non-null    float64
dtypes: float64(1), object(2)
memory usage: 3.6+ KB


**Observacions:**
- Ha aparegut un NaN per no poder convertir a float.

In [39]:
df_lloguer_stacked[df_lloguer_stacked["preu_mitja_m2"].isna()]

,Territori,any,preu_mitja_m2
128,Vallbona,2025,NaN


In [41]:
df_lloguer_raw[df_lloguer_raw["Territori"] == "Vallbona"]

,Territori,2015,2025
55,Vallbona,"6,7",-


**Observacions:**
- Valor = "-" per a 2025.

In [46]:
print("Territoris:\n", df_lloguer_stacked.Territori.unique())

Territoris:
 ['el Raval' 'el Barri Gòtic' 'la Barceloneta'
 'Sant Pere, Santa Caterina i la Ribera' 'el Fort Pienc'
 'la Sagrada Família' "la Dreta de l'Eixample"
 "l'Antiga Esquerra de l'Eixample" "la Nova Esquerra de l'Eixample"
 'Sant Antoni' 'el Poble Sec - AEI Parc Montjuïc'
 'la Marina del Prat Vermell - AEI Zona Franca' 'la Marina de Port'
 'la Font de la Guatlla' 'Hostafrancs' 'la Bordeta' 'Sants - Badal'
 'Sants' 'les Corts' 'la Maternitat i Sant Ramon' 'Pedralbes'
 'Vallvidrera, el Tibidabo i les Planes' 'Sarrià' 'les Tres Torres'
 'Sant Gervasi - la Bonanova' 'Sant Gervasi - Galvany'
 'el Putxet i el Farró' 'Vallcarca i els Penitents' 'el Coll' 'la Salut'
 'la Vila de Gràcia' "el Camp d'en Grassot i Gràcia Nova"
 'el Baix Guinardó' 'Can Baró' 'el Guinardó' "la Font d'en Fargues"
 'el Carmel' 'la Teixonera' 'Sant Genís dels Agudells' 'Montbau'
 "la Vall d'Hebron" 'la Clota' 'Horta' 'Vilapicina i la Torre Llobeta'
 'Porta' 'el Turó de la Peira' 'Can Peguera' 'la Guineueta' 'Ca

Es poden veure noms que no apareixen a la dimensió de barris. S' han de netejar.

In [78]:
df_lloguer_stacked["clean_territori"] = df_lloguer_stacked["Territori"].apply(
    lambda x: x.split("-")[0].strip().lower().replace(" ", "").replace("-", "") if "AEI" in x else x.strip().lower().replace(" ", "").replace("-", "")
    )
df_lloguer_stacked.head()

,Territori,any,preu_mitja_m2,clean_territori
0,el Raval,2015,11.1,elraval
1,el Barri Gòtic,2015,11.2,elbarrigòtic
2,la Barceloneta,2015,16.3,labarceloneta
3,"Sant Pere, Santa Caterina i la Ribera",2015,12.7,"santpere,santacaterinailaribera"
4,el Fort Pienc,2015,10.9,elfortpienc


In [85]:
# Obtenim el codi del barri a partir del nom
dim_barris["nom_barri_adpt"] = dim_barris["nom_barri"].apply(lambda x: x.strip().lower().replace(" ", "").replace("-", ""))

df_lloguer_codis = df_lloguer_stacked.merge(dim_barris[["codi_barri", "nom_barri", "nom_barri_adpt"]], left_on="clean_territori", right_on="nom_barri_adpt", how="left")\
    .drop(columns=["nom_barri_adpt", "clean_territori", "Territori", "nom_barri"])

# Mostrem si hi ha valors sense codi de barri
df_lloguer_codis[df_lloguer_codis["codi_barri"].isna()]

,any,preu_mitja_m2,codi_barri


No hi ha barris sense fer match

In [86]:
df_lloguer_25 = df_lloguer_codis[df_lloguer_codis["any"] == "2025"].drop(columns = ["any"])

df_lloguer_25.head()

,preu_mitja_m2,codi_barri
73,16.6,1
74,16.6,2
75,22.6,3
76,18.3,4
77,15.8,5


In [87]:
df_lloguer_15 = df_lloguer_codis[df_lloguer_codis["any"] == "2015"].drop(columns = ["any"])
df_lloguer_15.head()

,preu_mitja_m2,codi_barri
0,11.1,1
1,11.2,2
2,16.3,3
3,12.7,4
4,10.9,5


## Habitatges Turístics


In [90]:
with ZipFile("../data/raw/habitatge/Taula estadística - Nombre d’habitatges d’ús turístic.zip", "r") as zip_ref:
    zip_ref.extractall("../data/raw/habitatge/habitatges_turistics_zip_extract")

In [91]:
df = pd.read_csv("../data/raw/habitatge/habitatges_turistics_zip_extract/Taula estadística.csv")
df.head()

,Territori,Tipus de territori,31 Des. 2015,31 Des. 2023
0,el Raval,Barri,180,212.0
1,el Barri Gòtic,Barri,184,201.0
2,la Barceloneta,Barri,69,64.0
3,"Sant Pere, Santa Caterina i la Ribera",Barri,171,171.0
4,el Fort Pienc,Barri,343,333.0
